In [ ]:
import numpy as np
import scipy.linalg
import trovolut as trv
from pathlib import Path

## 1. Fermi-Hubbard Hamiltonian

$$H = -t \sum_{\langle ij\rangle,\sigma} \left(c^\dagger_{i\sigma} c_{j\sigma} + \text{h.c.}\right) + U \sum_i n_{i,\uparrow} n_{i,\downarrow}$$

In [ ]:
def get_hamiltonian():
    """Return Hamiltonian terms from a 1D Fermi-Hubbard model."""
    H = trv.qm.models.hubbard_1d(Lx=3, t_hop=1.0, U_onsite=4.0, boundary="open")
    return [t["coef"] * t["matrix"] for t in H.terms()]

hams = get_hamiltonian()
print(f"Model: 1D Fermi-Hubbard (3 sites, {int(np.log2(hams[0].shape[0]))} qubits, {len(hams)} terms)")

## 2. Define Constraints

We want the best accuracy possible, but it must run in under 30 seconds.

In [ ]:
time_hard = trv.eval.Time(max_seconds=30, hard=True, hard_penalty=-100.0)
accuracy_soft = trv.eval.Accuracy(weight=2.0, score_fn="inverse")

print(f"Constraint 1: {time_hard}\n             {time_hard.formula()}")
print(f"Constraint 2: {accuracy_soft}\n             {accuracy_soft.formula()}")

constraints = trv.eval.ConstraintSet([time_hard, accuracy_soft])
print(f"\nCombined:    {constraints.formula()}")

## 3. Configure Accuracy Measurement

Define how accuracy is computed - the setup and exact solution functions:

In [ ]:
def setup():
    """Return inputs for the candidate function: (hams, t, psi0)."""
    hams = get_hamiltonian()
    psi0 = np.zeros(hams[0].shape[0], dtype=complex)
    psi0[0] = 1.0
    return hams, 1.0, psi0

def exact_evolution(hams, t, psi0):
    """Compute exact evolution for reference."""
    return scipy.linalg.expm(-1j * sum(hams) * t) @ psi0

# Configure how accuracy is measured
constraints.with_accuracy(exact_fn=exact_evolution, setup_fn=setup).entry_point("run_dynamics")

## 4. Generate Files with Workflow

Pass `constraints` to auto-generate the evaluator:

In [ ]:
Path("opt").mkdir(exist_ok=True)

# Initial program (what OpenEvolve will optimize)
def run_dynamics(hams, t, psi0):
    """First-order Trotter evolution."""
    n_steps, dt = 100, t / 100
    psi = psi0.copy()
    for _ in range(n_steps):
        for H in hams:
            psi = scipy.linalg.expm(-1j * H * dt) @ psi
    return psi

# Build workflow with constraints attached
workflow = trv.filemaker.Workflow(
    [trv.filemaker.EVO_START, run_dynamics, trv.filemaker.EVO_END],
    out_path="opt/initial_program.py",
    imports=["import scipy.linalg"],
    constraints=constraints,  # enables build_evaluator()
)

# Build both files
init_path = workflow.build()
evaluator_path = workflow.build_evaluator()

print(f"Initial:   {init_path}")
print(f"Evaluator: {evaluator_path}")

## 5. Test Evaluator

In [ ]:
# Load the generated evaluator
exec(open(evaluator_path).read())

result = evaluate(str(init_path))
print(f"Score: {result['combined_score']:.4f}, All satisfied: {result['all_satisfied']}")
for name, info in result['details'].items():
    print(f"  {name}: {info['value']:.4f} → score={info['score']:.4f}")

## 6. Run OpenEvolve

In [ ]:
# Auto-generate prompt from constraints
print(constraints.generate_prompt())

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from openevolve import run_evolution
from openevolve.config import Config, LLMModelConfig, PromptConfig

config = Config()
config.prompt = PromptConfig(system_message=constraints.generate_prompt())
config.llm.models = [LLMModelConfig(name="gpt-4o")]

result = run_evolution(
    initial_program=str(init_path),
    evaluator=str(evaluator_path),
    iterations=10,
    config=config
)

In [ ]:
# Cleanup
import shutil
shutil.rmtree("opt", ignore_errors=True)